# Huấn luyện trên Kaggle / Colab

Notebook chạy `scripts/train.py` với cấu hình thí nghiệm trong `configs/`.
Sau khi huấn luyện, dùng `evaluate_cloud.ipynb` để đánh giá và
`result_analysis.ipynb` để so sánh kết quả.

- Kaggle: thêm code và dataset vào Input, bật GPU và Internet.
- Colab: đặt `DO_AN_AI_cloud.zip` và `PhenoBench_trainval.zip` trong
  `MyDrive/DO_AN_AI_cloud`, bật GPU.

Chạy các cell theo thứ tự từ trên xuống.


## 1. Chọn thí nghiệm

| Thí nghiệm | MODEL | LOSS |
|---|---|---|
| E1 | `deeplab` | `ce` |
| E2 | `segformer` | `ce` |
| E3 | Kiến trúc được chọn từ E1/E2 | `wce` |

`plan20` dùng ngân sách 20 epochs cho mỗi thí nghiệm. `original` đọc YAML gốc:
CE 3 epochs, DeepLab WCE 5 epochs, SegFormer WCE 10 epochs.
So sánh ảnh hưởng của loss cần cùng ngân sách huấn luyện.

Huấn luyện mới dùng `RESUME=False`. Khi tiếp tục thí nghiệm, giữ nguyên
model, loss, profile và tag rồi đặt `RESUME=True`. `RESTORE_FROM` là thư mục
chứa trực tiếp `last.pt`, `best.pt`, `config.yaml` và `training_log.csv` trong Input.
Để trống biến này nếu kết quả đã có ở thư mục output hoặc có thể tự tìm trong Input.


In [ ]:
MODEL = 'deeplab'
LOSS = 'ce'
PROFILE = 'plan20'
RUN_TAG = 'plan20_original_loop_v1'

RESUME = False
RESTORE_FROM = None
RUN_SMOKE = True


In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IS_KAGGLE = Path('/kaggle/input').is_dir()
if not IS_KAGGLE and not Path('/content').is_dir():
    raise RuntimeError('Notebook này dùng trên Kaggle hoặc Google Colab.')

BASE = Path('/kaggle/working') if IS_KAGGLE else Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/DO_AN_AI_cloud')
if not IS_KAGGLE:
    from google.colab import drive
    drive.mount('/content/drive')

print('Nền tảng:', 'Kaggle' if IS_KAGGLE else 'Colab')
subprocess.run(['nvidia-smi'], check=True)


## 2. Chuẩn bị code và thư viện

Code được đọc từ thư mục Input hoặc giải nén từ `DO_AN_AI_cloud.zip`.
Notebook cài `requirements-cloud.txt` và dùng PyTorch CUDA có sẵn của nền tảng.
Khởi động phiên mới khi cập nhật code.


In [ ]:
PROJECT = BASE / 'DO_AN_AI'
def extract_inside(archive_path, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        resolved = target.resolve()
        for member in archive.infolist():
            if not (target / member.filename).resolve().is_relative_to(resolved):
                raise ValueError('Unsafe archive path: ' + member.filename)
        archive.extractall(target)

if (PROJECT / 'scripts/train.py').is_file():
    print('Dùng source đã có:', PROJECT)
else:
    if IS_KAGGLE:
        expanded = sorted({p.parents[1] for p in Path('/kaggle/input').rglob('train.py')
                           if p.parent.name == 'scripts'
                           and (p.parents[1] / 'requirements-cloud.txt').is_file()})
        archives = sorted(Path('/kaggle/input').rglob('DO_AN_AI_cloud.zip'))
        if len(expanded) == 1:
            shutil.copytree(expanded[0], PROJECT)
        elif not expanded and len(archives) == 1:
            extract_inside(archives[0], BASE)
        else:
            raise RuntimeError('Add Input đúng một bộ code DO_AN_AI_cloud; bỏ các bộ trùng.')
    else:
        source_zip = DRIVE_ROOT / 'DO_AN_AI_cloud.zip'
        if not source_zip.is_file():
            raise FileNotFoundError('Upload code ZIP vào ' + str(source_zip))
        extract_inside(source_zip, BASE)
    assert (PROJECT / 'scripts/train.py').is_file(), 'Không thấy scripts/train.py'
print('Project:', PROJECT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(PROJECT / 'requirements-cloud.txt')], check=True)
check_gpu = (
    'import torch; print(torch.__version__); '
    'assert torch.cuda.is_available(); '
    'print(torch.cuda.get_device_name(0)); '
    'print(torch.cuda.get_device_properties(0).total_memory/1024**3)'
)
subprocess.run([sys.executable, '-c', check_gpu], check=True)


## 3. Kiểm tra dataset

Dataset gồm `train/images`, `train/semantics`, `val/images` và `val/semantics`.
Kiểm tra tên file ảnh và mask khớp nhau, đủ 1.407 cặp train và 772 cặp validation.
Kaggle đọc từ Input; Colab giải nén dữ liệu vào ổ của phiên chạy.


In [ ]:
def find_dataset(search_root):
    candidates = sorted({p.parent.parent for p in search_root.rglob('semantics')
                         if p.is_dir() and p.parent.name == 'train'
                         and (p.parent.parent / 'train/images').is_dir()
                         and (p.parent.parent / 'val/images').is_dir()
                         and (p.parent.parent / 'val/semantics').is_dir()})
    if len(candidates) > 1:
        raise RuntimeError('Nhiều dataset roots; chỉ attach một bộ PhenoBench: ' + str(candidates))
    return candidates[0] if candidates else None

if IS_KAGGLE:
    DATA_ROOT = find_dataset(Path('/kaggle/input'))
    if DATA_ROOT is None:
        local_data = BASE / 'phenobench_data'
        DATA_ROOT = find_dataset(local_data) if local_data.is_dir() else None
        if DATA_ROOT is None:
            archives = sorted(Path('/kaggle/input').rglob('PhenoBench_trainval.zip'))
            if len(archives) != 1:
                raise FileNotFoundError('Add Input PhenoBench_trainval.zip hoặc train/val đã giải nén.')
            extract_inside(archives[0], local_data)
            DATA_ROOT = find_dataset(local_data)
else:
    local_data = BASE / 'phenobench_data'
    DATA_ROOT = find_dataset(local_data) if local_data.is_dir() else None
    if DATA_ROOT is None:
        extract_inside(DRIVE_ROOT / 'PhenoBench_trainval.zip', local_data)
        DATA_ROOT = find_dataset(local_data)
assert DATA_ROOT is not None, 'Không tìm được dataset root'
for split, expected in [('train', 1407), ('val', 772)]:
    images = {p.name for p in (DATA_ROOT / split / 'images').glob('*.png')}
    masks = {p.name for p in (DATA_ROOT / split / 'semantics').glob('*.png')}
    assert len(images) == expected and images == masks, (split, len(images), len(masks))
    print(split, len(images), 'image/mask pairs')
print('Data root:', DATA_ROOT)


## 4. Đọc cấu hình

`plan20` dùng 20 epochs, batch size 4, crop 512 và learning rate theo kiến trúc.
Cell này đọc YAML và thay đường dẫn dataset, output cho môi trường cloud.

Trainer cập nhật trọng số sau mỗi batch. Loss của mỗi epoch là trung bình loss
các batch. Validation dùng ảnh nguyên và cùng batch size với train.


In [ ]:
import yaml

if MODEL not in ('deeplab', 'segformer') or LOSS not in ('ce', 'wce'):
    raise ValueError('Kiểm tra MODEL và LOSS ở bước 1.')
if PROFILE not in ('plan20', 'original'):
    raise ValueError('PROFILE phải là plan20 hoặc original.')
if not RUN_TAG.startswith(PROFILE) or not all(c.isalnum() or c in '-_' for c in RUN_TAG):
    raise ValueError('RUN_TAG cần bắt đầu bằng PROFILE, chỉ dùng chữ, số, dấu - hoặc _.')

if PROFILE == 'plan20':
    experiment = 'e3' if LOSS == 'wce' else 'e1' if MODEL == 'deeplab' else 'e2'
    config_path = PROJECT / f'configs/experiments/{experiment}_{MODEL}_{LOSS}.yaml'
else:
    config_path = PROJECT / f'configs/{MODEL}_{LOSS}.yaml'

output_base = BASE / 'outputs' if IS_KAGGLE else DRIVE_ROOT / 'outputs'
OUTPUT = output_base / RUN_TAG / MODEL / LOSS
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
config['data']['root'] = str(DATA_ROOT)
config['training']['output_dir'] = str(OUTPUT)
GENERATED_CONFIG = PROJECT / f'configs/cloud_{PROFILE}_{MODEL}_{LOSS}.yaml'
GENERATED_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Config:', config_path.relative_to(PROJECT))
print(yaml.safe_dump(config, sort_keys=False))


## 5. Chuẩn bị class weights và checkpoint

WCE với `class_weight: auto` tính trọng số từ mask của tập train.
Khi resume, sao chép cả thư mục thí nghiệm để giữ checkpoint tốt nhất và log.
Trainer kiểm tra cấu hình checkpoint trước khi tiếp tục.


In [ ]:
if LOSS == 'wce' and config['loss'].get('class_weight') == 'auto':
    subprocess.run([
        sys.executable, str(PROJECT / 'scripts/compute_class_stats.py'),
        '--data_root', str(DATA_ROOT), '--split', 'train', '--mode', 'merge',
        '--out', str(PROJECT / 'data/processed/class_stats.json'),
    ], cwd=PROJECT, check=True)

if RESUME and not (OUTPUT / 'last.pt').is_file():
    source = Path(RESTORE_FROM) if RESTORE_FROM else None
    if source is None and IS_KAGGLE:
        matches = []
        for path in Path('/kaggle/input').rglob('config.yaml'):
            saved = yaml.safe_load(path.read_text(encoding='utf-8')) or {}
            same_run = saved.get('training', {}).get('output_dir') == str(OUTPUT)
            if same_run and not saved.get('smoke_test') and (path.parent / 'last.pt').is_file():
                matches.append(path.parent)
        if len(matches) == 1:
            source = matches[0]
        else:
            raise RuntimeError(f'Điền RESTORE_FROM ở bước 1. Các bản tìm được: {matches}')
    if source is None:
        raise FileNotFoundError(f'Chưa có checkpoint ở {OUTPUT}. Kiểm tra Drive hoặc RESTORE_FROM.')
    if OUTPUT.exists() and any(OUTPUT.iterdir()):
        raise FileExistsError(f'Không copy đè thư mục đã có dữ liệu: {OUTPUT}')
    for name in ('last.pt', 'best.pt', 'config.yaml', 'training_log.csv'):
        if not (source / name).is_file():
            raise FileNotFoundError(source / name)
    saved = yaml.safe_load((source / 'config.yaml').read_text(encoding='utf-8'))
    if saved.get('smoke_test') or saved['training']['output_dir'] != str(OUTPUT):
        raise ValueError('Checkpoint khác RUN_TAG/output hoặc là smoke. Chọn đúng lần train.')
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, OUTPUT)

if RESUME:
    for name in ('last.pt', 'best.pt', 'config.yaml', 'training_log.csv'):
        if not (OUTPUT / name).is_file():
            raise FileNotFoundError(OUTPUT / name)
print('Chế độ:', 'resume từ last.pt' if RESUME else 'train mới từ pretrained')
print('Thư mục kết quả:', OUTPUT)


## 6. Kiểm tra pipeline và huấn luyện

Smoke test chạy 2 train batches và 1 validation batch để kiểm tra pipeline,
bộ nhớ GPU; kết quả lưu riêng vào thư mục có hậu tố `_smoke`.
Checkpoint smoke chỉ dùng để kiểm tra. Đặt `RUN_SMOKE=False` nếu lần kiểm tra
đã hoàn tất. Khi resume, notebook bỏ qua smoke test.

`epochs` trong YAML là tổng ngân sách huấn luyện. Với ngân sách 20 epochs,
checkpoint ở cuối epoch 7 sẽ tiếp tục từ epoch 8 đến epoch 20.
Thí nghiệm WCE bắt đầu từ pretrained, dùng `RESUME=False`.


In [ ]:
def run_training(smoke=False, resume=False):
    command = [
        sys.executable, '-u', str(PROJECT / 'scripts/train.py'),
        '--config', str(GENERATED_CONFIG), '--model_name', MODEL, '--device', 'cuda',
    ]
    if smoke:
        command.append('--smoke')
    if resume:
        command.extend(['--resume', str(OUTPUT / 'last.pt')])
    subprocess.run(command, cwd=PROJECT, check=True)

if RUN_SMOKE and not RESUME:
    run_training(smoke=True)


In [ ]:
run_training(resume=RESUME)


## 7. Lưu kết quả

- `best.pt`: checkpoint có validation loss thấp nhất, dùng để đánh giá.
- `last.pt`: checkpoint ở epoch hoàn tất gần nhất, dùng để resume.
- `config.yaml`, `training_log.csv`, `environment.json`, `requirements-lock.txt`:
  cấu hình, lịch sử loss và môi trường huấn luyện.

Kaggle xuất ZIP vào `/kaggle/working`; tải file từ bảng Output hoặc lưu phiên
kèm output trước khi đóng. `Save & Run All` chạy lại notebook trong phiên mới.
Colab ghi kết quả vào Google Drive.

Chỉ xuất kết quả khi tiến trình huấn luyện đã dừng. Nếu phiên bị ngắt,
checkpoint giữ trạng thái đến epoch đã hoàn tất.
Tiếp theo chạy [evaluate_cloud.ipynb](evaluate_cloud.ipynb).


In [ ]:
print('Experiment:', OUTPUT)
for name in ['best.pt', 'last.pt', 'training_log.csv', 'config.yaml',
             'environment.json', 'requirements-lock.txt']:
    path = OUTPUT / name
    assert path.is_file(), path
    print(name, round(path.stat().st_size / 1024**2, 2), 'MiB')
manifest = PROJECT / 'source_manifest.json'
if manifest.is_file():
    shutil.copy2(manifest, OUTPUT / 'source_manifest.json')
if IS_KAGGLE:
    results_zip = BASE / ('results_' + RUN_TAG + '_' + MODEL + '_' + LOSS + '.zip')
    with zipfile.ZipFile(results_zip, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(OUTPUT.parent).as_posix())
    print('ZIP:', results_zip)
    print('Tải trong bảng Output bên phải: mở /kaggle/working, tìm', results_zip.name)
    print('Chọn dấu ba chấm cạnh file, chọn Download.')
else:
    print('Đã lưu vào Google Drive:', OUTPUT)
